## 0 · Start With the Production Problem

> **The mission:** Riverside House needs an assistant for confidential publishing work without sending manuscripts to a public API. The first question is not *which fine-tuning method?* It is *what kind of behavior are we trying to create?*

| Production need | Best first move | Why |
| --- | --- | --- |
| Apply stable house style, sustain character voices and personas, preserve recurring storylines, or produce campaign copy in an established editorial voice | Fine-tuning | The behavior should persist across many prompts; examples can teach prose, tone, structure, and response habits. |
| Answer from changing HR policies, legal terms, contracts, benefits guides, product catalogs, or other documents that must remain current and citable | RAG | The facts belong in an external source of truth that can be updated, retrieved, quoted, permission-filtered, and audited. |
| Perform a task the current model already handles when given a clear instruction and a few examples | Prompting first | No model update or retrieval system is justified until the simpler control fails. |
| Use a stable brand persona while answering from current approved documents | Fine-tuning plus RAG | Fine-tuning supplies the durable behavior; retrieval supplies the changing evidence. |

Fine-tuning is a poor database: facts become difficult to update, cite, delete, or permission-filter. RAG is a poor substitute for learned behavior: retrieved passages can provide facts, but they do not reliably install an editor's voice, a character persona, or a response contract.

**Riverside's three teaching goals:**

1. Practice Riverside's unpublished prose, characters, and storylines.
2. Practice a bounded editor request.
3. Practice choosing the draft an editor would keep.

This notebook changes the training-example shape for those three goals. The later RAG chapter handles current, citable document knowledge rather than asking model weights to serve as a policy store.

# LLM Fine-Tuning Deep Dive, Part 1 of 3: What Should the Model Learn?

> **The story:** Riverside changes the model's training experience to teach three different behaviors: continued pretraining practices its prose, SFT practices an assistant contract, and DPO practices editorial preference.
>
> **Where you are:** The transformer chapters explained next-token prediction. Part 1 builds intuition for **what behavior the examples teach**; Part 2 explores **where the update is stored**; Part 3 is the evaluation notebook.

## The Aria Training Story

> **Fictional corpus context:** This notebook uses Riverside's complete 40-chapter generation-ship novel *The Weight of Distant Light*. Aria Voss is a Systems Maintenance technician aboard the *Meridian's Promise* whose discovery of an artificial prime-number signal begins the novel's first-contact arc with the ancient distributed intelligence later known as the Choir.

All 40 chapters supply teaching examples. That choice gives the tiny, compute-bounded model the strongest available training signal and keeps this notebook focused on how each objective works.

## Why There Is No Holdout Here

This is a mechanism notebook, not a capability study. A 135M checkpoint, one short novel, and ten-step CPU runs cannot support a useful claim about unseen behavior; withholding chapters would weaken an already small training signal while still producing a noisy result. Consequently, examples and outputs in Part 1 are **in-sample demonstrations only**. They show how data enters each objective, not how a candidate behaves beyond that data.

Part 3 owns evaluation and requires a separately versioned dataset that was never used to build the training examples.

| Part | Question |
| --- | --- |
| 1 - this notebook | What training experience teaches the desired behavior? |
| [2 - parameter strategy](02-llm-finetuning-parameter-techniques.ipynb) | Where can that update live, and what does each mechanism change? |
| [3 - evaluation and decision](03-llm-finetuning-comparison-and-decision.ipynb) | What does independent evidence support? |

Continued pretraining and SFT start from separate pinned base models; the preference stage continues from the SFT adapter. Checkpoints and training-provenance manifests are written under `checkpoints/llm-finetuning/<profile>/`.

> **Boundary:** fine-tuning changes persistent behavior. A later retrieval chapter supplies current, citable facts.

## Fine-Tuning Roadmap: Start Here

Read the arc vertically: Part 1 changes the learning objective, Part 2 changes the parameter mechanism, and Part 3 introduces evidence and decisions.

```mermaid
flowchart TD
    Start["Pinned SmolLM2<br/>fluent, but unfamiliar with Riverside"]

    subgraph Data["Part 1 - choose the learning objective"]
        direction TB
        C1["Continued pretraining<br/>practice Riverside prose"]
        C2["SFT<br/>practice a request/response contract"]
        C3["Preference alignment<br/>practice chosen over rejected"]
        C1 --> C2 --> C3
    end

    subgraph Params["Part 2 - choose the parameter mechanism"]
        direction TB
        C4["Full fine-tuning"]
        C5["Partial freezing"]
        C6["LoRA"]
        C7["QLoRA + quantization"]
        C4 --> C5 --> C6 --> C7
    end

    Start --> C1
    C3 --> Saved["Training artifacts<br/>full 40-chapter corpus"]
    Saved --> C4
    C7 --> Evaluate["Part 3<br/>metrics, comparison, gates, decisions"]
```

> The arrows are a learning sequence, not literal checkpoint ancestry. Parts 1 and 2 demonstrate mechanisms on the complete corpus; only Part 3 may interpret measurements or make decisions.

## Prerequisite Bridge: From Encoder-Decoder Attention to a Decoder-Only Assistant

The transformer foundations introduced three useful shapes: an **encoder** reads an entire input, a **decoder** predicts the next token while respecting a causal mask, and an **encoder-decoder** model lets a decoder attend to an encoded source through cross-attention. Riverside's assistant uses the decoder-only choice: at each turn, the user's instruction, any supplied scene, and the completion form one growing token sequence.

| Foundation | Role in this chapter | Why Riverside needs it |
| --- | --- | --- |
| Causal decoder | The selected SmolLM2 profile predicts the next token | It can continue prose and answer prompts from one left-to-right context |
| Training objective | Labels say which next tokens should become more likely | Continued pretraining, SFT, and preference learning change what Riverside teaches the decoder |
| Encoder / retrieval later | Encodes a query and passages for matching | It finds current, citable evidence instead of asking the generator to remember every fact |

So this notebook changes **how a decoder-only model behaves**. It does not turn the model into a dependable catalog lookup system.

The underlying mechanics are owned by the prerequisite notebooks:

- [Transformers Part 12](../02-transformers/02-decoder-only-language-model.ipynb#part-12---the-causal-triangle-and-the-accumulation-tower) explains causal visibility.
- [Transformers Part 8](../02-transformers/02-decoder-only-language-model.ipynb#part-8---mini-language-model-training--inference) owns decoder-only training, including the [per-position loss microscope](../02-transformers/02-decoder-only-language-model.ipynb#per-position-loss-one-sequence-many-lessons) and [one complete backward/update trace](../02-transformers/02-decoder-only-language-model.ipynb#one-backward-pass-many-token-lessons-one-update).
- [PyTorch RNN Bridge Part 4](../01-rnns/01-pytorch-rnn-bridge.ipynb#part-4--explicit-sequence-training-and-gradient-clipping) owns shifted sequence loss, backpropagation, and optimizer mechanics.
- [Encoder-Decoder Part 5](../02-transformers/03-encoder-decoder-and-cross-attention.ipynb#part-5--full-encoder-decoder-training) owns teacher-forced seq2seq training and cross-attention.

This notebook assumes those mechanics and focuses on the fine-tuning decision: **which examples and labels teach the behavior Riverside needs?**

> **Implementation preview:** the objective and parameter strategy are separate choices. This notebook uses full fine-tuning for the continued-pretraining demonstration, then small LoRA adapters for SFT and the final preference update so the runs fit local hardware. Treat LoRA here as a small trainable correction attached to a frozen base; Part 2 opens that black box and compares it with full and partial fine-tuning.

## Learning Route

1. Pin the base revision, seed, artifact paths, and complete Aria corpus.
2. Build raw-text examples and inspect how continued pretraining labels every real token.
3. Build request/response demonstrations and inspect response-only SFT masking.
4. Build chosen/rejected comparisons and develop PPO-versus-DPO intuition.
5. Run each short training mechanism and save training-provenance artifacts.
6. Continue to Part 2 for full, frozen, LoRA, and QLoRA parameter intuition.
7. Continue to Part 3, where evaluation begins.

**Optional depth:** token-level mechanics live in the prerequisite notebooks linked below. The resumable-job appendix packages the same full-corpus training paths without interpreting the resulting behavior.

In [ ]:
from pathlib import Path

# Resolve the notebook directory once so corpus and checkpoint paths do not depend on kernel cwd.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

REPO_ROOT = _notebook_dir.parents[2]
CONTENT_DIR = _notebook_dir / "content"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
ARIA_NOVEL_DIR = CONTENT_DIR / "the-weight-of-distant-light"

if not ARIA_NOVEL_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the Aria corpus at {ARIA_NOVEL_DIR}. Open and run this notebook from "
        "learning/genai/03-llm-finetuning so the committed content directory resolves correctly."
    )

ARIA_CHAPTER_FILES = sorted(ARIA_NOVEL_DIR.glob("chapter-*.txt"))
if not ARIA_CHAPTER_FILES:
    raise ValueError("The Aria corpus does not contain any chapter files")

# This compute-bounded concept notebook uses every available chapter for training.
ARIA_TRAIN_FILES = ARIA_CHAPTER_FILES


def load_paragraphs(chapter_files, min_len=200):
    """Load qualifying paragraphs from explicit, provenance-preserving chapter files."""
    paragraphs = []
    for path in chapter_files:
        text = path.read_text(encoding="utf-8")
        for paragraph in text.split("\n\n"):
            paragraph = paragraph.strip().replace("\n", " ")
            if len(paragraph) >= min_len:
                paragraphs.append(paragraph)
    return paragraphs


sample_paragraphs = load_paragraphs(ARIA_TRAIN_FILES[:2])
print(f"Aria corpus used for training: {len(ARIA_TRAIN_FILES)} chapters")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"First training paragraph:\n\n{sample_paragraphs[0][:421]} ...")

## Motivation: See the Starting Behavior Before Naming a Technique

Riverside begins with three distinct behavior gaps:

| Behavior gap | Illustrative request | What the example reveals |
| --- | --- | --- |
| Unfamiliar domain prose | Continue a passage containing Riverside-only names and relationships | The model can write fluent English while inventing the story world |
| Unpracticed response contract | `Answer in one sentence and stop.` | General instruction ability does not guarantee this exact house workflow |
| Unexpressed editorial preference | Compare two valid answers to the same request | Demonstrations alone do not state which valid draft an editor prefers |

Start with the prose gap. The base model has never seen Riverside's manuscripts, so names in a prompt can guide a plausible guess without installing the recurring relationships or style.

That motivates the first training experience:

> The model already practices English; continued pretraining lets it practice Riverside text.

---

### Setting Up the Shared Teaching Model

The three sections use pinned checkpoints and fixed illustrative prompts so the mechanics remain reproducible.

### Pick Capacity From the Hardware, Keep the Objective Fixed

The notebook chooses a matched base/instruction pair from one model family:

| Runtime | Continued-pretraining base | SFT/DPO base | Purpose |
| --- | --- | --- | --- |
| CPU | `HuggingFaceTB/SmolLM2-135M` | `HuggingFaceTB/SmolLM2-135M-Instruct` | Keep every mechanism runnable on ordinary hardware. |
| CUDA below 64 GiB | `HuggingFaceTB/SmolLM2-360M` | `HuggingFaceTB/SmolLM2-360M-Instruct` | Give a typical GPU a stronger baseline without making the multi-model comparison impractical. |
| CUDA with at least 64 GiB | `HuggingFaceTB/SmolLM2-1.7B` | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | Prefer the strongest same-family teaching checkpoint when memory supports the complete arc. |

The objective, prompt contract, LoRA targets, data split, and evidence gates do not change between profiles. Artifacts are written to a profile-specific directory because adapters and full checkpoints cannot be moved safely between model sizes.

In [ ]:
import hashlib
import json
import random
from importlib.metadata import version

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA_AVAILABLE else 0.0
)

if not CUDA_AVAILABLE:
    MODEL_PROFILE = "cpu-small-135m"
    CPT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
    CPT_MODEL_REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    INSTRUCT_MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
elif GPU_MEMORY_GIB < 64:
    MODEL_PROFILE = "gpu-balanced-360m"
    CPT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
    CPT_MODEL_REVISION = "f8027fd0eaeea54caa13c31d31b9fdc459c38b49"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
    INSTRUCT_MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
else:
    MODEL_PROFILE = "gpu-quality-1.7b"
    CPT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B"
    CPT_MODEL_REVISION = "effd688a12921b4cc83e3312b6feb579f70f9c71"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    INSTRUCT_MODEL_REVISION = "31b70e2e869a7173562077fd711b654946d38674"

MODEL_NAME = INSTRUCT_MODEL_NAME
MODEL_REVISION = INSTRUCT_MODEL_REVISION
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
EXPERIMENT_SEED = 2026
DEMO_TRAIN_STEPS = 10
DEMO_DPO_STEPS = 10
CPU_BUDGET_MINUTES = 360
GPU_BUDGET_MINUTES = 20

PROFILE_CHECKPOINT_DIR = CHECKPOINT_DIR / "llm-finetuning" / MODEL_PROFILE
CPT_CHECKPOINT_DIR = PROFILE_CHECKPOINT_DIR / "non-instruction-full"
SFT_CHECKPOINT_DIR = PROFILE_CHECKPOINT_DIR / "instruction-lora"
DPO_CHECKPOINT_DIR = PROFILE_CHECKPOINT_DIR / "preference-dpo"

random.seed(EXPERIMENT_SEED)
set_seed(EXPERIMENT_SEED)


def _file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_artifact_manifest(
    stage,
    output_dir,
    training_files,
    training_args,
    extra=None,
    model_name=MODEL_NAME,
    model_revision=MODEL_REVISION,
):
    """Record immutable model, full-corpus, package, and training provenance beside an artifact."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    manifest = {
        "stage": stage,
        "model_profile": MODEL_PROFILE,
        "model": {"id": model_name, "revision": model_revision},
        "seed": EXPERIMENT_SEED,
        "training_files": [
            {"path": path.relative_to(REPO_ROOT).as_posix(), "sha256": _file_sha256(path)}
            for path in training_files
        ],
        "training_arguments": training_args.to_dict(),
        "packages": {
            package: version(package)
            for package in ("torch", "transformers", "datasets", "peft", "trl")
        },
        "extra": extra or {},
    }
    manifest_path = output_dir / "experiment-manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print(f"Wrote provenance manifest: {manifest_path}")
    return manifest_path


training_budget_minutes = GPU_BUDGET_MINUTES if CUDA_AVAILABLE else CPU_BUDGET_MINUTES
print(f"Model profile: {MODEL_PROFILE}")
print(f"CUDA memory: {GPU_MEMORY_GIB:.1f} GiB")
print(f"CPT model: {CPT_MODEL_NAME}@{CPT_MODEL_REVISION[:8]}")
print(f"SFT/DPO model: {INSTRUCT_MODEL_NAME}@{INSTRUCT_MODEL_REVISION[:8]}")
print(f"Profile checkpoints: {PROFILE_CHECKPOINT_DIR}")
print(f"Training budget on this device: {training_budget_minutes} minutes")
if not CUDA_AVAILABLE:
    print(
        "CPU disclaimer: the 135M model and ten-step runs are for learning the training "
        "mechanisms, not for demonstrating production-quality prose or generalization."
    )

### How to Read the Small CPU Run

The 135M CPU profile is intentionally mechanism-sized. Ten updates are enough to expose tokenization, masking, gradients, adapters, trainer state, and checkpoint boundaries, but not enough to establish behavior beyond the exercise.

Treat every generated sample in Parts 1 and 2 as an intuition-building illustration from a model trained on the complete corpus. Evaluation begins in Part 3.

> **PyTorch → Keras:** `torch.cuda.is_available()` — checks whether a CUDA-capable GPU is visible to PyTorch and returns a bool; the result picks the `device` string (`"cuda"` or `"cpu"`) that every tensor and model call below is pinned to via `.to(device)`. **Keras/TF equivalent:** `tf.config.list_physical_devices('GPU')` — TensorFlow auto-places ops on any visible GPU without needing an explicit device string threaded through the code, so most Keras code skips this check entirely; `tf.device(...)` exists for the rare case you want to force placement.

In [ ]:
device = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"Using device: {device} ({MODEL_PROFILE})")

### Loading the Tokenizers

Continued pretraining and instruction-tuned generation use different checkpoints, so each path loads the tokenizer pinned to its own model revision. The selected SmolLM2 checkpoints share a 49,152-token vocabulary, and their EOS token also serves as the padding token.

This section stays at the token level: how text becomes IDs, why whitespace changes token boundaries, and how padding is excluded from loss. Role-based instruction prompts and response-only supervision begin in the SFT section.

> **PyTorch → Keras:** `AutoTokenizer.from_pretrained(...)` loads the same checkpoint-matched tokenizer for either framework. Tokenization is framework-agnostic; the split between PyTorch and TensorFlow begins when the resulting arrays become framework tensors.

In [ ]:
# Load the tokenizer pinned to each model path before comparing their token contracts.
cpt_tokenizer = AutoTokenizer.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
)
if cpt_tokenizer.pad_token is None:
    cpt_tokenizer.pad_token = cpt_tokenizer.eos_token

tokenizer = AutoTokenizer.from_pretrained(
    INSTRUCT_MODEL_NAME,
    revision=INSTRUCT_MODEL_REVISION,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"CPT tokenizer vocabulary: {len(cpt_tokenizer):,}")
print(f"CPT EOS/pad: {cpt_tokenizer.eos_token!r} / {cpt_tokenizer.pad_token!r}")
print(f"Instruction-checkpoint tokenizer vocabulary: {len(tokenizer):,}")
print(f"Instruction-checkpoint EOS/pad: {tokenizer.eos_token!r} / {tokenizer.pad_token!r}")

### Seeing the Vocabulary in Action

Byte-level BPE can represent arbitrary text, but common spans receive compact tokens while rare names or
unusual Unicode sequences split into several pieces. Whitespace is context: tokenizing `"signal"` and
`" signal"` can produce different IDs because a space may be merged with neighboring bytes.

The code below prints IDs, raw tokenizer tokens, and decoded pieces for each example. Decoding each ID is
the portable way to make spaces and newlines visible; raw token strings are implementation details and
should not be treated as a universal notation.


> **PyTorch → Keras:** `tokenizer.encode(word)` / `tokenizer.convert_ids_to_tokens(ids)` — converts raw text to integer token IDs (and back to readable BPE-piece strings) using the framework-agnostic tokenizer loaded above; no tensors are created yet, just plain Python lists. **Keras/TF equivalent:** identical call — `AutoTokenizer` isn't PyTorch- or TF-specific, so a Keras/TF version of this notebook would use this exact same code; only the downstream model call (`TFAutoModelForCausalLM` vs. `AutoModelForCausalLM`) would differ.

In [ ]:
# One example each of a noun, proper noun, verb, and adjective -- all pulled from Riverside's own
# sci-fi opening line, so these are words this notebook already leans on elsewhere.
example_words = {
    "noun": "signal",
    "proper noun": "Aria",
    "verb": "stared",
    "adjective": "distant",
}

for part_of_speech, word in example_words.items():
    ids_alone = tokenizer.encode(word)  # tokenize the word standalone (no leading space)
    ids_mid_sentence = tokenizer.encode(" " + word)  # tokenize as it would appear mid-sentence
    print(f"{part_of_speech.upper()}: {word!r}")
    print(
        f"  as the first word of a text  : ids={ids_alone}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_alone)}"
    )
    print(
        f"  mid-sentence (' {word}')".ljust(31) + f": ids={ids_mid_sentence}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_mid_sentence)}"
    )
    print()

print(
    "'\u0120' at the start of a token marks a leading space -- it's why the same word can tokenize "
    "differently depending on where it appears in a sentence."
)


> **You may wonder:** since `Aria` splits into two tokens and appears constantly in this corpus, why
> not just train the tokenizer on Riverside's own text and merge it into a dedicated token? Two
> reasons this is out of scope for fine-tuning: extending the vocabulary adds a new, untrained row
> to the embedding matrix and output head, and filling that row in with a meaningful representation
> is itself a training problem, not something fine-tuning does for free. And splitting `Aria` into
> two tokens doesn't stop the model from learning what it means -- it can still learn to associate
> that two-token pattern with everything fine-tuning teaches it about her; it just costs two sequence
> positions instead of one, a small efficiency tax, not a correctness problem. The actual gap the
> rest of this notebook closes is that the model has never seen who Aria Voss is, not how her name
> happens to be tokenized.



> **A related question:** what happens with a word the tokenizer has rarely encountered? Byte-level
> BPE does not need an unknown-word vocabulary entry: when no longer merge matches, it falls back to smaller
> byte-derived pieces. An uncommon name such as `Itzpapalotl` therefore remains representable, although it
> usually consumes more tokens than a frequent word. Fine-tuning can improve how the model uses that sequence,
> but it does not add a new vocabulary row unless the tokenizer and embedding matrix are explicitly resized.


### Loading the Base Model

`base_model` is the actual pretrained neural network -- a checkpoint-defined number of real weights downloaded from the
Hugging Face hub, moved onto whichever device we resolved above via `.to(device)`. This untouched
checkpoint is the "before" every fine-tuning technique in this notebook is compared against.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` — downloads the weights selected by `MODEL_NAME` into a PyTorch `nn.Module` and moves every parameter tensor onto `device` (CPU or GPU) in place. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` — loads the same checkpoint into a `tf.keras.Model` instead; TensorFlow doesn't need an explicit `.to(device)` call since ops are placed on available devices automatically (or via a `tf.device(...)` context).

In [ ]:
# Load the untouched instruction baseline used by SFT, DPO, and general-knowledge probes.
base_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL_NAME,
    revision=INSTRUCT_MODEL_REVISION,
).to(device)
model_parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
decoder_blocks = base_model.model.layers
n_blocks = len(decoder_blocks)
hidden_size = base_model.config.hidden_size
print(
    f"Loaded {INSTRUCT_MODEL_NAME}@{INSTRUCT_MODEL_REVISION[:8]}: "
    f"{model_parameter_count:,} parameters, {n_blocks} decoder blocks, "
    f"hidden size {hidden_size}."
)

### A Fixed Test Prompt for Before/After Comparisons

`PROMPT` is the one fixed test sentence reused throughout the notebook so "before" vs. "after"
fine-tuning comparisons are always apples-to-apples. It's pulled straight from the sci-fi corpus so
a model that has actually absorbed the catalog has a real chance of continuing it in-world.


In [ ]:
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus

### A Reusable `generate()` Helper

Hugging Face returns the prompt and completion in one token sequence. The helper records the prompt length, slices `output[prompt_length:]`, and decodes only the new tokens so every comparison shows the model's actual continuation.

It also calls `.strip()` because the first generated token may carry leading whitespace. All later candidates use this same helper, so output formatting cannot masquerade as a model difference.

> **PyTorch → Keras:** `model.eval()` / `torch.no_grad()` / `model.generate()` — `.eval()` switches dropout/batchnorm-style layers to inference mode, `torch.no_grad()` disables gradient tracking to save memory during inference, and `.generate()` runs HuggingFace's autoregressive sampling loop (nucleus sampling here via `top_p`/`temperature`). **Keras/TF equivalent:** `TFAutoModelForCausalLM.generate()` — the same HuggingFace `.generate()` API exists on TF models with identical sampling arguments; TF's analog of "eval mode" is passing `training=False` (implicit inside `.generate()`), and there's no separate "no_grad" context since calling a `tf.keras.Model` outside a `GradientTape` block already skips gradient recording.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base model, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.

    Notes
    -----
    A real example from this notebook's own `PROMPT` (15 tokens) makes both
    lines concrete. Asking for `max_new_tokens=15` returns `out` with shape
    `(1, 30)` -- the 15 prompt tokens plus 15 new ones, concatenated. Decoding
    all 30 without slicing prints the prompt right back before the answer:

        'Aria Voss stared at the signal counting itself out in prime numbers
         and began to ponder the question, what was it that she had to do?'

    `out[0][prompt_len:]` (`prompt_len = 15` here) drops the first 15 tokens so
    only the new continuation gets decoded. But decoding *just* those 15 new
    tokens gives:

        ' began to ponder the question, what was it that she had to do?'

    -- note the stray leading space: the decoded continuation can begin with whitespace carried by its first
    token, so the raw decoded string may start with a space. `.strip()`
    removes it, along with any trailing whitespace/newlines near the end.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(
        device
    )  # use the same tokenizer to tokenize the prompt and convert it to tensor
    prompt_len = inputs["input_ids"].shape[1]  # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # stochastic → varied output
            top_p=0.9,  # nucleus sampling: top 90% mass
            temperature=0.8,  # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )

    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return (
        completion
        if completion
        else "[model stopped immediately — sampled EOS as first token]"
    )


print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")
print(f"Completion: {generate(base_model, PROMPT, 20)}")

### Pick Capacity From the Hardware, Keep the Objective Fixed

The notebook chooses a matched base/instruction pair from one model family:

| Runtime | Continued-pretraining base | SFT/DPO base | Purpose |
| --- | --- | --- | --- |
| CPU | `HuggingFaceTB/SmolLM2-135M` | `HuggingFaceTB/SmolLM2-135M-Instruct` | Keep every mechanism runnable on ordinary hardware. |
| CUDA below 64 GiB | `HuggingFaceTB/SmolLM2-360M` | `HuggingFaceTB/SmolLM2-360M-Instruct` | Expose the same mechanisms with a larger model. |
| CUDA with at least 64 GiB | `HuggingFaceTB/SmolLM2-1.7B` | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | Use the strongest same-family teaching checkpoint when memory permits. |

The objective, prompt contract, LoRA targets, complete-corpus policy, and training steps do not change between profiles. Artifacts are written to a profile-specific directory because adapters and full checkpoints cannot move safely between model sizes.

### Transformer Mechanics Live Upstream

A causal-LM training batch follows one compact contract: token IDs enter the decoder, each position predicts the next token, padding labels use `-100`, and backpropagation updates whichever parameters remain trainable.

This chapter does not re-derive that contract. Use the prerequisite links above for causal masks, per-position cross-entropy, gradient flow, and optimizer updates. From here onward, every code path answers a training question:

- Which Riverside text becomes continued-pretraining data?
- Which prompt tokens must SFT hide from the loss?
- Which chosen/rejected pairs express editor preference?
- Which model state is saved for the next chapter?

The next section begins at that fine-tuning-specific boundary.

In [ ]:
# Shared analysis imports used by later fine-tuning diagnostics.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch.nn.functional as F

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Shared analysis libraries loaded.")

---

## Choose the Training Objective From the Required Behavior

Each methodology supplies a different training experience. Choose the one whose examples most closely resemble what the production model must practice; do not build a mandatory technique chain.

| Required persistent behavior | Training experience | Riverside example | Other examples |
| --- | --- | --- | --- |
| Absorb recurring language, entities, and prose patterns | Continued pretraining on raw domain text | Practice Aria, Wren, the Choir, and Riverside's narrative style | Medical terminology, source-code conventions, specialist jargon |
| Follow a repeatable request/response contract | SFT on demonstrations | Continue a scene in exactly one sentence and stop | Structured extraction, support templates, campaign copy in a brand persona |
| Prefer one valid answer over another | DPO on chosen/rejected pairs | Advance the scene instead of restating it | Prefer concise edits, safer refusals, or editor-approved tone |
| Learn from outcomes produced by the changing model | PPO with fresh rollouts and feedback | Not needed for Riverside's fixed pair dataset | Tool-use success, simulator reward, live environment outcomes |

```mermaid
flowchart TD
    Need["Behavior the model must practice"] --> Kind{"What kind of training signal matches it?"}
    Kind -->|"raw domain sequence"| CPT["Continued pretraining"]
    Kind -->|"demonstrated response"| SFT["SFT"]
    Kind -->|"fixed comparison"| DPO["DPO"]
    Kind -->|"fresh outcome feedback"| PPO["PPO"]
    CPT --> Artifact["Train and save artifact"]
    SFT --> Artifact
    DPO --> Artifact
    PPO --> Artifact
    Artifact --> Part3["Part 3: evaluate on independent evidence"]
```

Part 1 asks only how the objective works. Part 3 asks whether the resulting behavior is useful.

## Concept 1: Continued Pretraining - Practice Riverside Prose

The base model has not practiced Aria-specific names, relationships, and prose patterns. Continued pretraining keeps the ordinary next-token objective and changes only the text stream: raw paragraphs from all 40 chapters become training sequences.

| What this experience can teach | What it cannot directly teach |
| --- | --- |
| Domain vocabulary, recurring entities, prose patterns | How to obey a user request |
| Which token sequences resemble Riverside prose | When to stop or return a required format |
| A domain-oriented starting point for later adaptation | Which of two acceptable answers an editor prefers |

The runnable example updates all model weights so the learning signal is easy to inspect. Part 2 later separates that objective from the choice of which parameters may move.

> **Bridge to SFT:** practicing manuscript continuation does not practice the one-sentence interaction contract. The next section changes the example shape from raw text to request/response demonstrations.

### Code Walkthrough: `tokenize_causal()` - Preparing Text for Next-Token Prediction

This is the first point where raw Aria paragraphs become the fixed-length integer tensors a transformer consumes. `tokenize_causal()` is defined immediately before the dataset mapping that needs it. The guarded resumable-job appendix reuses the same function; SFT and DPO use response-aware contracts instead.

```python
def tokenize_causal(examples, tokenizer, max_length=64):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_overflowing_tokens=True,
    )
    tokens["labels"] = [
        [(token_id if mask == 1 else -100) for token_id, mask in zip(ids, attention)]
        for ids, attention in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    return tokens
```

**Arguments:**

| Argument | Type | Purpose |
| --- | --- | --- |
| `examples` | `dict` (HF batch) | A batch with a `"text"` column containing paragraphs from the 36 Aria training chapters. |
| `tokenizer` | `PreTrainedTokenizer` | The tokenizer loaded from the pinned model revision. |
| `max_length` | `int`, default `64` | Fixed sequence length. Long paragraphs produce overflow chunks; short final chunks are padded. |

**What it returns:** tokenizer output (`input_ids`, `attention_mask`) plus `labels`. Each real token id becomes its own causal-LM label, while every padding position becomes `-100`, PyTorch's ignore index.

`return_overflowing_tokens=True` matters: without it, truncation would discard paragraph tails. With it, the tail becomes another fixed-length training example. The final short chunk remains present with its padding labels masked.

### Optional Depth: Long Documents, Truncation, and Packing

Plain `truncation=True` is a paper cutter: without overflow handling, an 800-token example capped at 512 contributes only its first 512 tokens.

This notebook's `tokenize_causal()` also sets `return_overflowing_tokens=True`, so a long paragraph becomes multiple fixed-length chunks instead of silently losing its tail. The final short chunk is padded and its padding labels are masked.

| Training type | Common strategy | Trade-off |
| --- | --- | --- |
| SFT | Truncate or separately budget prompt and completion | Preserves pair structure, but an overlong response may still lose its tail |
| Continued pretraining | Overflow chunks or pack documents into fixed blocks | Preserves more text, but block boundaries weaken cross-boundary context |

```text
BLOCK A: [ Token 0 ... Token 127 ]
BLOCK B: [ Token 128 ... ]  <- attention starts again here
```

The first tokens in Block B cannot attend to Block A even when they continue the same paragraph. Production pipelines may use document-aware packing, block-diagonal attention, best-fit grouping, or local overlap to manage that trade-off.

For this teaching run, overflow chunks keep every paragraph tail visible while preserving a simple fixed-length loss mask.

### From Raw Paragraphs to Causal-LM Training Blocks

The next cell performs only the continued-pretraining data transformation:

1. tokenize every qualifying paragraph from all 40 chapters;
2. preserve long paragraph tails as overflow blocks;
3. copy real token IDs into `labels` so every non-padding position practices next-token prediction; and
4. replace padding labels with `-100` so padding contributes no gradient.

This is the core intuition: continued pretraining does not need instructions, answers, preferences, or scores. It needs domain sequences and the ordinary causal next-token objective.

> **PyTorch to Keras:** `Dataset.map()` prepares the token fields consumed by HuggingFace `Trainer`; a Keras pipeline would perform the same transformation with `tf.data.Dataset.map()` before `model.fit()`.

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments


def tokenize_causal(examples, tokenizer_to_use, max_length=64):
    """Create next-token labels while preserving long paragraphs as fixed-length chunks."""
    tokens = tokenizer_to_use(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_overflowing_tokens=True,
    )
    tokens["labels"] = [
        [(token_id if mask == 1 else -100) for token_id, mask in zip(ids, attention)]
        for ids, attention in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    return tokens


non_inst_paragraphs = load_paragraphs(ARIA_TRAIN_FILES)
non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})
non_inst_tokenized = non_inst_dataset.map(
    lambda examples: tokenize_causal(examples, cpt_tokenizer),
    batched=True,
    remove_columns=["text"],
)

print(
    f"Continued-pretraining source: {len(non_inst_paragraphs):,} paragraphs from "
    f"all {len(ARIA_TRAIN_FILES)} Aria chapters"
)
print(f"Tokenized training data: {len(non_inst_tokenized):,} 64-token chunks")

The full corpus is tokenized. Load two fresh copies of the selected plain SmolLM2 checkpoint: one remains unchanged so the weight-movement exercise can show where training wrote updates, and the other receives ten optimizer steps.

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(CPT_MODEL_NAME, revision=CPT_MODEL_REVISION).to(device)` loads an independent copy of the selected plain causal checkpoint, while `p.numel()` counts its scalar parameters. A TensorFlow version would use `TFAutoModelForCausalLM.from_pretrained(...)` and `model.count_params()`; TensorFlow places operations on available devices without PyTorch's `.to(device)` call.

In [ ]:
import gc

# Make this cell safe to rerun without retaining stale full-FT state.
for model_name in ("trainer_full", "full_ft_model", "cpt_base_model"):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cpt_base_model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
).to(device)
full_ft_model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
).to(device)

print(
    f"Loaded two independent {CPT_MODEL_NAME}@{CPT_MODEL_REVISION[:8]} copies: "
    f"{sum(parameter.numel() for parameter in full_ft_model.parameters()):,} parameters each."
)

### Configuring and Running the Trainer

`TrainingArguments` + `Trainer` is HuggingFace's standard training loop -- it handles batching, the forward/backward pass, and the optimizer step described earlier in this notebook, so we do not write that loop by hand. `DEMO_TRAIN_STEPS = 10` and `learning_rate=5e-5` keep this CPU demonstration bounded; a real Riverside training run would choose its budget from measured convergence. `trainer_full.train()` runs all ten real optimizer steps used by every later comparison in this notebook.

> **PyTorch → Keras:** `TrainingArguments(...)` / `Trainer(model=..., args=..., train_dataset=...)` / `trainer_full.train()` / `full_ft_model.save_pretrained(...)` — configures and runs HuggingFace's full PyTorch training loop (batching, forward/backward passes, optimizer steps, logging) in one `.train()` call, then serializes the fine-tuned weights + config to disk. **Keras/TF equivalent:** `model.compile(optimizer=..., loss=...)` + `model.fit(dataset, epochs=...)` — the direct Keras analog of configuring + running training; `save_pretrained(...)` has an identically-named method on `TFPreTrainedModel` subclasses, so the checkpoint-saving line itself would be unchanged in a TF version.

In [ ]:
# Ten real optimizer updates keep the mechanism runnable on every selected profile.
training_args_full = TrainingArguments(
    output_dir=str(CPT_CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=1,
    save_strategy="no",
    learning_rate=5e-5,
    seed=EXPERIMENT_SEED,
    data_seed=EXPERIMENT_SEED,
    report_to="none",
)

trainer_full = Trainer(
    model=full_ft_model,
    args=training_args_full,
    train_dataset=non_inst_tokenized,
)
trainer_full.train()
full_ft_model.save_pretrained(CPT_CHECKPOINT_DIR)
cpt_tokenizer.save_pretrained(CPT_CHECKPOINT_DIR)
write_artifact_manifest(
    stage="continued-pretraining-full",
    output_dir=CPT_CHECKPOINT_DIR,
    training_files=ARIA_TRAIN_FILES,
    training_args=training_args_full,
    extra={"objective": "causal language modeling", "parameter_strategy": "full"},
    model_name=CPT_MODEL_NAME,
    model_revision=CPT_MODEL_REVISION,
)
print("Saved continued-pretraining checkpoint and training-provenance manifest.")

### Weight Movement Layer by Layer

A single aggregate can hide where full fine-tuning changed the network. The cell below samples the same number of individual absolute weight deltas from every transformer block and gives **each block its own panel**.

Figures contain at most 10 panels, so the runtime-reported decoder blocks are paginated automatically. Every panel uses the same y-axis scale: a quiet block therefore cannot look as active as a strongly moving block merely because its axis was automatically rescaled.

> **What to look for:** Compare the mean and maximum in each panel title, then inspect the shape. Long regions near zero mean many sampled weights barely moved; spikes identify sampled weights with larger changes. These are sampled magnitudes, not a claim that one block alone stores the learned behavior.

In [ ]:
# Reload the saved checkpoint and compare its real weight movement with the pinned CPT base.
import gc

base_state = dict(cpt_base_model.named_parameters())
ft_trace_model = AutoModelForCausalLM.from_pretrained(CPT_CHECKPOINT_DIR).to("cpu")

SAMPLES_PER_BLOCK = 500
PANELS_PER_FIGURE = 10
N_COLUMNS = 2

all_deltas = []
for block_index in range(len(cpt_base_model.model.layers)):
    block_deltas = []
    for name, parameter in ft_trace_model.named_parameters():
        if f"model.layers.{block_index}." in name:
            delta = (parameter.data.cpu() - base_state[name].data.cpu()).abs().flatten()
            block_deltas.append(delta)
    if block_deltas:
        combined = torch.cat(block_deltas)
        stride = max(1, len(combined) // SAMPLES_PER_BLOCK)
        all_deltas.append(combined[::stride][:SAMPLES_PER_BLOCK].float().numpy())

global_ymax = max(float(block.max()) for block in all_deltas)
y_limit = global_ymax * 1.05 if global_ymax > 0 else 1e-9

for page_start in range(0, len(all_deltas), PANELS_PER_FIGURE):
    page = all_deltas[page_start : page_start + PANELS_PER_FIGURE]
    row_count = (len(page) + N_COLUMNS - 1) // N_COLUMNS
    figure, axes = plt.subplots(
        row_count,
        N_COLUMNS,
        figsize=(14, 2.6 * row_count),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()

    for panel_index, block_values in enumerate(page):
        block_index = page_start + panel_index
        weight_indices = np.arange(len(block_values))
        axis = axes[panel_index]
        axis.plot(weight_indices, block_values, linewidth=0.7, color="steelblue")
        axis.fill_between(weight_indices, block_values, alpha=0.12, color="steelblue")
        axis.set_title(
            f"Transformer block {block_index}  "
            f"(mean={block_values.mean():.2e}, max={block_values.max():.2e})",
            fontsize=9,
        )
        axis.set_ylim(0, y_limit)
        axis.grid(alpha=0.2, axis="y")

    for unused_axis in axes[len(page) :]:
        unused_axis.set_visible(False)

    page_end = page_start + len(page) - 1
    figure.suptitle(
        f"Full Fine-Tuning Weight Movement: Blocks {page_start}-{page_end}",
        fontsize=12,
        fontweight="bold",
    )
    figure.supxlabel(f"Sampled weight index ({SAMPLES_PER_BLOCK} weights per block)")
    figure.supylabel("|W_after - W_before|")
    plt.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.show()

peak_block = max(range(len(all_deltas)), key=lambda index: all_deltas[index].mean())
min_block = min(range(len(all_deltas)), key=lambda index: all_deltas[index].mean())
print(
    f"Mean |delta W| by block - min: block {min_block} "
    f"({all_deltas[min_block].mean():.4e}), max: block {peak_block} "
    f"({all_deltas[peak_block].mean():.4e})."
)

del ft_trace_model
gc.collect()
print("Freed the trace model; the checkpoint and manifest remain on disk.")

### Continued-Pretraining Intuition Check

The important objects are now visible:

- raw Riverside paragraphs became fixed-length causal blocks;
- every real token became a next-token target;
- padding was masked;
- all model weights were trainable; and
- the layer-by-layer trace shows that full fine-tuning writes changes throughout the network.

This section establishes the training mechanism only. Evaluation begins in Part 3.

## Concept 2: Supervised Fine-Tuning - Specialize the Request/Response Contract

Continued pretraining practices manuscript prose, not an assistant contract. SFT changes the shape of the teaching example: each example now pairs a request with the response Riverside wants the assistant to produce.

Riverside supplies demonstrations with two roles:

- **Request:** continue the supplied Aria context in exactly one sentence and stop.
- **Desired response:** the first sentence of the next real manuscript paragraph.

### Instruction Prompts Begin Here

`render_instruction()` uses SmolLM2's native chat template to serialize the system instruction, user request, and assistant response. For training, the complete sequence is visible to the model, but only the assistant suffix contributes to loss:

```text
[prompt tokens: visible, labels=-100] [assistant response: visible, labels=token IDs] [padding: labels=-100]
```

This response-only supervision is the central SFT intuition. The model sees the request as context without being trained to reproduce it.

All 40 chapters supply adjacent-paragraph demonstrations, alternating two equivalent request phrasings. The ten-step LoRA run exists to expose the data contract, adapter mechanics, and artifact boundary. Whether the resulting assistant follows the contract on independent requests is an evaluation question reserved for Part 3.

> **Bridge to preference alignment:** SFT demonstrates one desired response. Preference data becomes useful when several responses are plausible and an editor can say which one is better.

This teaching run uses LoRA to fit local hardware. Part 2 opens that parameter mechanism.

> **PyTorch → Keras:** `from peft import LoraConfig, get_peft_model, TaskType` — imports HuggingFace's PEFT library, which wraps a PyTorch model's targeted `nn.Linear` layers with low-rank adapter matrices and freezes everything else; the actual wrapping happens a few cells down. **Keras/TF equivalent:** there is no first-party `peft` support for `TFPreTrainedModel`s — the common Keras/TF pattern for parameter-efficient tuning is manual layer freezing (`layer.trainable = False` on all but the last few layers) rather than LoRA adapters, since PEFT's LoRA implementation is PyTorch-only.

In [ ]:
import gc
import re

from peft import LoraConfig, PeftModel, TaskType, get_peft_model

# The CPT models are no longer needed after the mechanism exercises.
for model_name in (
    "trainer_full",
    "full_ft_model",
    "cpt_base_model",
    "base_model",
    "base_state",
):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed continued-pretraining models before LoRA SFT.")

SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."


def render_instruction(instruction, response=None):
    """Render the native SmolLM2 chat contract used by SFT and DPO."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction.strip()},
    ]
    if response is None:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    messages.append({"role": "assistant", "content": response.strip()})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


SFT_TRAIN_TASKS = (
    "Continue this Aria scene in exactly one sentence and stop.",
    "Write one sentence that continues this passage, then stop.",
)
INSTRUCTION_TASK = SFT_TRAIN_TASKS[0]


def first_complete_sentence(text, max_text_tokens=24):
    """Return one complete sentence that fits safely inside the assistant suffix budget."""
    text = text.strip()
    match = re.search(r"^.*?[.!?](?:[\"']?)(?=\s|$)", text)
    sentence = match.group(0).strip() if match else text
    words = sentence.split()

    while words:
        candidate = " ".join(words).rstrip(" ,;:-.!?") + "."
        token_count = len(tokenizer(candidate, add_special_tokens=False)["input_ids"])
        if token_count <= max_text_tokens:
            return candidate
        words.pop()
    raise ValueError("Could not construct a complete bounded response sentence")


def build_instruction_pairs(chapter_files, tasks, max_pairs=None):
    """Build adjacent-paragraph SFT pairs from explicit chapter files and request phrasings."""
    pairs = []
    for path in chapter_files:
        paragraphs = [
            paragraph.strip().replace("\n", " ")
            for paragraph in path.read_text(encoding="utf-8").split("\n\n")
            if len(paragraph.strip()) > 200
        ]
        for pair_index, (context, next_paragraph) in enumerate(
            zip(paragraphs, paragraphs[1:])
        ):
            task = tasks[pair_index % len(tasks)]
            instruction = f"{task}\n\nContext:\n{context}"
            pairs.append(
                {"instruction": instruction, "response": first_complete_sentence(next_paragraph)}
            )
            if max_pairs is not None and len(pairs) >= max_pairs:
                return pairs
    return pairs

### Where These Instruction Pairs Come From

The runnable path constructs demonstrations from adjacent paragraphs across all 40 chapters:

```text
paragraph i                         -> request context
first sentence of paragraph i + 1 -> desired assistant response
```

Two equivalent request phrasings alternate through the corpus. The pair builder is deliberately mechanical so the learner can inspect exactly how prose becomes instruction data.

| Role | What this notebook uses |
| --- | --- |
| Pair construction | Python adjacency logic over the complete Aria corpus |
| Model being specialized | Pinned SmolLM2 instruction checkpoint |
| Optional synthetic-pair generator | Not used |

This extraction teaches one narrow contract. It does not establish editing quality, generalization, or production readiness; those concepts begin in Part 3.

### Tokenizing With the Prompt-Mask Pattern

Token-aware budgeting keeps at most 64 native prompt tokens and at most 32 native assistant-suffix tokens inside 96 positions. Prompt and padding labels are `-100`, and exactly one assistant EOS token is supervised.


> **PyTorch → Keras:** `tokenize_instruction()` — builds `labels` as a copy of the tokenized `input_ids`, then overwrites *both* the prompt-token positions and the padding positions with `-100`, so a cross-entropy loss with `ignore_index=-100` (used earlier in the notebook) only ever grades the completion tokens. **Keras/TF equivalent:** the same masking logic — a Keras/TF version would build an analogous `labels` array with `-100` (or `0` plus a matching `sample_weight` mask, since TF's `SparseCategoricalCrossentropy` has no built-in `ignore_index`) at prompt+padding positions; the tokenization itself is identical since `AutoTokenizer` is framework-agnostic.

In [ ]:
_SFT_CONTEXT_MARKER = "\n\nContext:\n"


def _sft_token_ids(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def _sft_prefix_encoding(text):
    if tokenizer.is_fast:
        return tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    return tokenizer(text, add_special_tokens=False)


def _sft_token_prefix_text(text, encoding, token_count):
    if token_count == 0:
        return ""
    offsets = encoding.get("offset_mapping")
    if offsets is not None:
        return text[: offsets[token_count - 1][1]].rstrip()
    return tokenizer.decode(
        encoding["input_ids"][:token_count],
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    ).rstrip()


def _sft_largest_fitting_prefix(text, max_source_tokens, build_candidate):
    encoding = _sft_prefix_encoding(text)
    low = 0
    high = min(len(encoding["input_ids"]), max_source_tokens)
    best = None

    while low <= high:
        token_count = (low + high) // 2
        prefix = _sft_token_prefix_text(text, encoding, token_count)
        candidate = build_candidate(prefix)
        if candidate is None:
            high = token_count - 1
        else:
            best = candidate
            low = token_count + 1

    return best


def _sft_bounded_prompt(instruction, prompt_max_length):
    task, marker, context = instruction.strip().partition(_SFT_CONTEXT_MARKER)
    if not marker:
        raise ValueError("Instruction is missing the expected Context section")

    context = context.strip()
    instruction_prefix = task + marker

    def build_candidate(bounded_context):
        bounded_instruction = instruction_prefix + bounded_context
        prompt_text = render_instruction(bounded_instruction)
        prompt_ids = _sft_token_ids(prompt_text)
        if len(prompt_ids) <= prompt_max_length:
            return bounded_instruction, prompt_text, prompt_ids
        return None

    result = _sft_largest_fitting_prefix(
        context, prompt_max_length, build_candidate
    )
    if result is None:
        raise ValueError("The fixed instruction template exceeds the prompt budget")
    return result


def _sft_bounded_assistant_suffix(
    bounded_instruction,
    prompt_text,
    prompt_ids,
    response,
    assistant_max_length,
):
    if tokenizer.eos_token is None or tokenizer.eos_token_id is None:
        raise ValueError("The tokenizer must define an EOS token")

    def build_candidate(bounded_response):
        full_text = render_instruction(bounded_instruction, bounded_response)
        assert full_text.startswith(prompt_text), (
            "Native chat template did not preserve the exact prompt text prefix"
        )

        assistant_text = full_text[len(prompt_text) :]
        eos_offset = assistant_text.rfind(tokenizer.eos_token)
        if eos_offset < 0:
            raise ValueError("Native assistant rendering did not contain EOS")

        through_eos_text = prompt_text + assistant_text[
            : eos_offset + len(tokenizer.eos_token)
        ]
        through_eos_ids = _sft_token_ids(through_eos_text)
        if through_eos_ids[: len(prompt_ids)] != prompt_ids:
            return None

        assistant_ids = through_eos_ids[len(prompt_ids) :]
        if (
            len(assistant_ids) <= assistant_max_length
            and assistant_ids
            and assistant_ids[-1] == tokenizer.eos_token_id
            and assistant_ids.count(tokenizer.eos_token_id) == 1
        ):
            return assistant_ids
        return None

    result = _sft_largest_fitting_prefix(
        response.strip(), assistant_max_length, build_candidate
    )
    if result is None:
        raise ValueError("No stable native assistant suffix fits the assistant budget")
    return result


def tokenize_instruction(
    example,
    max_length=96,
    prompt_max_length=64,
    assistant_max_length=32,
):
    if prompt_max_length + assistant_max_length != max_length:
        raise ValueError("Prompt and assistant budgets must sum to max_length")
    if tokenizer.pad_token_id is None:
        raise ValueError("The tokenizer must define a padding token")

    bounded_instruction, prompt_text, prompt_ids = _sft_bounded_prompt(
        example["instruction"], prompt_max_length
    )
    assistant_ids = _sft_bounded_assistant_suffix(
        bounded_instruction,
        prompt_text,
        prompt_ids,
        example["response"],
        assistant_max_length,
    )

    active_ids = prompt_ids + assistant_ids
    padding_length = max_length - len(active_ids)
    if padding_length < 0:
        raise AssertionError("Bounded prompt and assistant suffix exceed max_length")

    input_ids = active_ids + [tokenizer.pad_token_id] * padding_length
    attention_mask = [1] * len(active_ids) + [0] * padding_length
    labels = [-100] * len(prompt_ids) + assistant_ids + [-100] * padding_length

    assert assistant_ids[-1] == tokenizer.eos_token_id
    assert assistant_ids.count(tokenizer.eos_token_id) == 1
    assert len(input_ids) == len(attention_mask) == len(labels) == max_length
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


### Building the Full-Corpus Instruction Dataset

Build demonstrations from every Aria chapter, then tokenize each row with the same 64-token prompt budget and 32-token assistant budget. The assertions inspect the training contract: prompt labels are masked, completion labels are active, padding labels are masked, and exactly one EOS token ends the supervised suffix.

In [ ]:
instruction_pairs = build_instruction_pairs(ARIA_TRAIN_FILES, SFT_TRAIN_TASKS)
print(f"Built {len(instruction_pairs):,} Aria SFT pairs from all 40 chapters")

instruction_dataset = Dataset.from_list(instruction_pairs)
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction,
    remove_columns=["instruction", "response"],
)

for row in instruction_tokenized:
    input_ids = row["input_ids"]
    attention_mask = row["attention_mask"]
    labels = row["labels"]
    assert len(input_ids) == len(attention_mask) == len(labels) == 96
    prompt_boundary = next(index for index, label in enumerate(labels) if label != -100)
    active_length = sum(attention_mask)
    active_response_labels = labels[prompt_boundary:active_length]
    decoded_response = tokenizer.decode(
        active_response_labels[:-1],
        skip_special_tokens=False,
    ).strip()
    assert all(label == -100 or mask == 1 for label, mask in zip(labels, attention_mask))
    assert all(label == -100 for label in labels[:prompt_boundary])
    assert all(label != -100 for label in active_response_labels)
    assert all(label == -100 for label in labels[active_length:])
    assert prompt_boundary <= 64
    assert len(active_response_labels) <= 32
    assert active_response_labels.count(tokenizer.eos_token_id) == 1
    assert active_response_labels[-1] == tokenizer.eos_token_id
    assert decoded_response.endswith(".")

print(
    f"All {len(instruction_tokenized):,} SFT rows preserve the 64/32 boundary "
    "and one complete supervised response sentence."
)

### A Quick LoRA Preview for This SFT Run

SFT defines **what behavior is taught**. LoRA only changes **where the update is stored**: `get_peft_model()` freezes the base and adds small trainable correction matrices to selected attention projections.

That is enough detail for this chapter. The next code cell uses LoRA so the SFT run fits local hardware; Part 2 derives the low-rank path, measures its parameter budget, and inspects the real matrices.

> **PyTorch → Keras:** `LoraConfig(...)` and `get_peft_model(...)` target the four SmolLM2 attention
projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and report the resulting trainable fraction.
PEFT's wrapping is PyTorch-specific; a Keras implementation needs a compatible low-rank layer wrapper or
manual custom layers rather than coarse whole-layer freezing.


In [ ]:
# Rank-8 adapters on all four attention projections.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()

### Training and Saving the Adapter

Same `Trainer` pattern as continued pretraining, just with the LoRA-wrapped model, the prompt-masked
dataset, and a higher learning rate (`2e-4` vs. `5e-5`) -- LoRA needs a higher LR since it's only
updating a tiny slice of parameters.


> **PyTorch → Keras:** `Trainer(model=instruct_lora_model, ...)` / `trainer_instruct.train()` / `instruct_lora_model.save_pretrained(...)` — the same HuggingFace `Trainer` pattern as the earlier full fine-tuning run, just pointed at the LoRA-wrapped model and the prompt-masked instruction dataset, with a higher learning rate since only the small adapter matrices are being updated. **Keras/TF equivalent:** `model.fit(dataset, epochs=...)` — as with the earlier full fine-tuning cell, a Keras/TF version would call `.fit()` on the (layer-frozen) model instead of `Trainer.train()`; `save_pretrained()` again has an identically-named counterpart on `TFPreTrainedModel`.

In [ ]:
# Short LoRA run to expose response-masked SFT and adapter saving.
training_args_instruct = TrainingArguments(
    output_dir=str(SFT_CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=1,
    save_strategy="no",
    learning_rate=2e-4,
    seed=EXPERIMENT_SEED,
    data_seed=EXPERIMENT_SEED,
    report_to="none",
)

trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained(SFT_CHECKPOINT_DIR)
tokenizer.save_pretrained(SFT_CHECKPOINT_DIR)
write_artifact_manifest(
    stage="supervised-fine-tuning-lora",
    output_dir=SFT_CHECKPOINT_DIR,
    training_files=ARIA_TRAIN_FILES,
    training_args=training_args_instruct,
    extra={
        "objective": "response-masked causal language modeling",
        "parameter_strategy": "lora",
        "training_tasks": list(SFT_TRAIN_TASKS),
    },
)
print("Saved SFT LoRA adapter and training-provenance manifest.")

### Instruction Tuning, Recapped

The preceding cells implement the SFT mechanism:

1. adjacent Aria paragraphs become request/response demonstrations;
2. all 40 chapters contribute teaching examples;
3. the request remains visible while its labels are masked with `-100`;
4. only the assistant suffix, including one EOS token, contributes gradient; and
5. a LoRA adapter stores the update while the base remains frozen.

```text
Continued pretraining: [labels for every real text token ......] [pad: -100]
Instruction tuning:   [prompt: -100 ........] [completion labels] [pad: -100]
```

The conceptual change is the supervision boundary. Part 3 later asks whether that boundary produced useful behavior on independent requests.

### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only **padding** was masked, and every real token was active. Here, the **bounded prompt context is masked**, while the **bounded assistant suffix is supervised**. The model is penalized only for the assistant suffix, including exactly one EOS token, never for reproducing the prompt it was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it through the real `tokenize_instruction()` used for training, and colors every token position by what the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)

# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

# Visualize the mask layout as a single color-coded strip (gray=prompt, green=completion, white=padding)
fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)

# Legend entries matching each color band in the strip above
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {render_instruction(example_pair['instruction'])[:80]!r}...")
print(f"Completion text: {example_pair['response'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

## Concept 3: Preference Learning - Valid Is Not Yet Preferred

SFT can demonstrate a valid one-sentence response, but several responses can satisfy the same contract while differing sharply in editorial value.

```text
Context: Aria watched the prime-number signal repeat across node seventeen.

Chosen:   Aria isolated the transmission and called Wren to verify that the pattern was artificial.
Rejected: Aria considered the signal again and remained where she was while nothing changed.
```

Both responses are grammatical and bounded. The first advances the scene; the second restates it. That distinction motivates a different example shape:

| Field | Meaning |
| --- | --- |
| `prompt` | One shared request and context |
| `chosen` | The response an editor would keep |
| `rejected` | A plausible response the editor would discard |

An SFT demonstration says, "practice this response." A preference pair says, "for this prompt, practice choosing this response over that one." It does not assign a universal quality score.

Before naming an algorithm, the training design needs to decide whether feedback is fixed or continually produced by the changing model. PPO handles fresh interaction loops; DPO uses an existing offline comparison set.

### PPO: Learn From Fresh Drafts and Fresh Feedback

Use PPO when the important evidence is produced by the changing model itself: a tool succeeds or fails, a simulated action earns a reward, or reviewers continually score new drafts.

```mermaid
flowchart LR
    Policy["Current policy"] --> Drafts["Generate fresh drafts or actions"]
    Drafts --> Feedback["Collect reward or outcome feedback"]
    Feedback --> Judge["Estimate which actions helped"]
    Judge --> Cautious["Make a bounded update"]
    Cautious --> Check["Check retained behavior and safety"]
    Check --> Policy
```

The intuition is restraint. One batch of fresh experience is useful but noisy, so PPO lets good evidence influence the model without allowing that batch to justify an unlimited jump. Then the changed policy gathers new experience and repeats the loop.

Riverside does not need this machinery: its editors already supplied a fixed comparison dataset, and there is no environment to explore. That makes the offline DPO path below the closer match.

#### Image Generation Prompt: PPO

> Create a landscape 16:9 editorial infographic titled "PPO: Learn From Fresh Experience". Show a clockwise five-stage loop with large numbered panels: 1) a current language model produces a new Riverside manuscript draft, 2) an editor or tool environment returns feedback, 3) a reward model and value estimate interpret the result, 4) a safety brake labeled "bounded update" prevents one batch from pushing too far, 5) the updated model generates the next fresh draft. Include a small frozen SFT reference model beside the loop as an anchor. Use a sophisticated publishing-workflow aesthetic, off-white paper background, charcoal text, deep red and teal accents, crisp vector shapes, readable labels, no equations, no probability ratios, no decorative gradients, and no 3D effects. Add a bottom callout: "Use PPO when new actions create new evidence."

### DPO: Learn Directly From Fixed Editorial Comparisons

Use DPO when the training data already contains prompt, chosen response, and rejected response triples. Riverside has exactly that example shape, so it can skip the reward model, value model, and fresh-rollout loop.

```mermaid
flowchart LR
    Pair["Full-corpus pair<br/>prompt + chosen + rejected"] --> Copies["Start with two SFT copies"]
    Copies --> Ref["Frozen reference<br/>keeps the starting behavior fixed"]
    Copies --> Policy["Trainable policy"]
    Ref --> Contrast["Contrast chosen and rejected support"]
    Policy --> Contrast
    Contrast --> Update["Update the policy toward chosen"]
    Update --> Artifact["Save the DPO adapter"]
```

The frozen reference is part of the training mechanism, not an evaluation baseline. It anchors the policy to its SFT starting point while the chosen/rejected examples supply the update direction.

| Use PPO when... | Use DPO when... |
| --- | --- |
| The current policy must generate new trajectories and receive new feedback | The comparison dataset is already fixed |
| Tool results or environment outcomes matter | A smaller, auditable offline run matters |
| Fresh outcomes should continually enter training | Existing chosen/rejected pairs express the desired preference |

DPO cannot invent preferences absent from its pair dataset. This notebook uses all 40 chapters to construct a compact set of length-matched training triples and focuses on how those triples enter `DPOTrainer`.

#### Image Generation Prompt: DPO

> Create a landscape 16:9 editorial infographic titled "DPO: Learn From Fixed Comparisons". On the left, show one Riverside editing card containing a manuscript context and two complete one-sentence drafts: a green "CHOSEN: advances the scene" draft and a muted red "REJECTED: repeats the setup" draft. In the center, split one SFT checkpoint into two identical model blocks: a gray frozen reference with a lock icon and a teal trainable policy with an update icon. On the right, show the policy update moving toward the chosen draft and ending in a saved adapter artifact. Add a small crossed-out strip for "no reward model, no value model, no rollout loop". Use an elegant publishing-house visual language, off-white paper background, charcoal typography, deep red and teal accents, crisp vector arrows, readable labels, no equations, no numeric score derivations, no decorative gradients, and no 3D effects.

In [ ]:
# DPO Step 1: build length-matched preference triples from the complete corpus.
DPO_MAX_LENGTH = 128
DPO_MAX_PROMPT_TOKENS = 64
DPO_MAX_RESPONSE_TOKENS = DPO_MAX_LENGTH - DPO_MAX_PROMPT_TOKENS - 1
DPO_TASK = "Continue this Aria scene in exactly one sentence and stop."
STALLING_RESPONSE = (
    "Aria considered the same situation again and repeated what she already knew while "
    "remaining exactly where she was as nothing in the scene changed."
)


def _token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def _bounded_prompt(context):
    """Keep as much Aria context as fits beside the native chat-template overhead."""
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    context_ids = context_ids[:DPO_MAX_PROMPT_TOKENS]
    while context_ids:
        bounded_context = tokenizer.decode(context_ids, skip_special_tokens=True).strip()
        instruction = f"{DPO_TASK}\n\nContext:\n{bounded_context}"
        prompt = render_instruction(instruction)
        if _token_count(prompt) <= DPO_MAX_PROMPT_TOKENS:
            return instruction, prompt
        context_ids = context_ids[:-1]
    raise ValueError("Chat-template overhead leaves no room for DPO context tokens")


def _render_response_suffix(instruction, prompt, response):
    """Render one complete response with exactly one terminal EOS marker."""
    full_text = render_instruction(instruction, response)
    if not full_text.startswith(prompt):
        raise ValueError("Instruction template did not preserve the DPO prompt prefix")
    response_suffix = full_text[len(prompt) :]
    eos_index = response_suffix.find(tokenizer.eos_token)
    if eos_index == -1:
        raise ValueError("Rendered response does not contain the tokenizer EOS marker")
    return response_suffix[: eos_index + len(tokenizer.eos_token)]


def _matched_stalling_suffix(instruction, prompt, target_tokens):
    """Find a complete repetitive sentence with exactly the chosen suffix token count."""
    words = STALLING_RESPONSE.rstrip(".").split()
    for word_count in range(4, len(words) + 1):
        candidate = " ".join(words[:word_count]) + "."
        suffix = _render_response_suffix(instruction, prompt, candidate)
        if _token_count(suffix) == target_tokens:
            return suffix
    return None


def build_preference_pairs(chapter_files, max_pairs):
    """Build real-next-sentence versus repetitive-stall pairs from explicit chapters."""
    pairs = []
    for path in chapter_files:
        paragraphs = [
            paragraph.strip().replace("\n", " ")
            for paragraph in path.read_text(encoding="utf-8").split("\n\n")
            if len(paragraph.strip()) > 200
        ]
        for context, next_paragraph in zip(paragraphs, paragraphs[1:]):
            instruction, prompt = _bounded_prompt(context)
            chosen_text = first_complete_sentence(
                next_paragraph,
                max_text_tokens=DPO_MAX_RESPONSE_TOKENS - 4,
            )
            chosen = _render_response_suffix(instruction, prompt, chosen_text)
            chosen_tokens = _token_count(chosen)
            rejected = _matched_stalling_suffix(
                instruction,
                prompt,
                target_tokens=chosen_tokens,
            )
            if rejected is None:
                continue
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
            if len(pairs) >= max_pairs:
                return pairs
    return pairs


preference_pairs = build_preference_pairs(ARIA_TRAIN_FILES, max_pairs=32)
if len(preference_pairs) < 8:
    raise ValueError("Not enough clean, length-matched DPO pairs were produced")

for pair in preference_pairs:
    prompt_tokens = _token_count(pair["prompt"])
    chosen_tokens = _token_count(pair["chosen"])
    rejected_tokens = _token_count(pair["rejected"])
    assert chosen_tokens == rejected_tokens
    assert prompt_tokens + chosen_tokens <= DPO_MAX_LENGTH
    assert pair["chosen"].removesuffix(tokenizer.eos_token).rstrip().endswith(".")
    assert pair["rejected"].removesuffix(tokenizer.eos_token).rstrip().endswith(".")

example_preference = preference_pairs[0]
print(f"Built {len(preference_pairs)} full-corpus DPO training triples")
print("\n=== One length-matched editorial comparison ===")
print(f"Prompt tokens:   {_token_count(example_preference['prompt'])}")
print(f"Chosen tokens:   {_token_count(example_preference['chosen'])}")
print(f"Rejected tokens: {_token_count(example_preference['rejected'])}")
print(f"Chosen:   {example_preference['chosen'][:220]!r}")
print(f"Rejected: {example_preference['rejected'][:220]!r}")

In [ ]:
# Inspect triplets across the training pool instead of trusting one convenient example.
preview_indices = sorted(
    {
        0,
        len(preference_pairs) // 5,
        2 * len(preference_pairs) // 5,
        3 * len(preference_pairs) // 5,
        4 * len(preference_pairs) // 5,
        len(preference_pairs) - 1,
    }
)

for preview_number, pair_index in enumerate(preview_indices, start=1):
    pair = preference_pairs[pair_index]
    prompt_context = pair["prompt"].split("Context:\n", maxsplit=1)[-1]
    prompt_context = prompt_context.split(tokenizer.eos_token, maxsplit=1)[0].strip()
    chosen_text = pair["chosen"].removesuffix(tokenizer.eos_token).strip()
    rejected_text = pair["rejected"].removesuffix(tokenizer.eos_token).strip()

    print("\n" + "=" * 88)
    print(f"Representative training triplet {preview_number} (dataset index {pair_index})")
    print(f"Context : {prompt_context[-180:]}")
    print(f"Chosen  : {chosen_text}")
    print(f"Rejected: {rejected_text}")
    print(
        f"Tokens  : chosen={_token_count(pair['chosen'])}, "
        f"rejected={_token_count(pair['rejected'])}"
    )

print(f"\nThe DPO trainer will consume {len(preference_pairs)} fixed comparison triples.")

In [ ]:
# DPO Step 2: train the policy against a frozen SFT reference.
import gc

from peft import PeftModel
from trl import DPOConfig, DPOTrainer

DPO_BETA = 0.1

for model_name in (
    "trainer_instruct",
    "instruct_lora_model",
    "instruct_base",
    "dpo_trainer",
    "dpo_policy_model",
    "dpo_policy_base",
    "dpo_reference_model",
    "dpo_reference_base",
):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed completed SFT state before loading the DPO policy and reference.")

dpo_policy_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
).to(device)
dpo_reference_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
).to(device)
dpo_policy_model = PeftModel.from_pretrained(
    dpo_policy_base,
    SFT_CHECKPOINT_DIR,
    is_trainable=True,
).to(device)
dpo_reference_model = PeftModel.from_pretrained(
    dpo_reference_base,
    SFT_CHECKPOINT_DIR,
    is_trainable=False,
).to(device)
dpo_reference_model.eval()
for parameter in dpo_reference_model.parameters():
    parameter.requires_grad_(False)

dpo_dataset = Dataset.from_list(preference_pairs)
dpo_args = DPOConfig(
    output_dir=str(DPO_CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    max_length=DPO_MAX_LENGTH,
    max_steps=DEMO_DPO_STEPS,
    learning_rate=5e-5,
    beta=DPO_BETA,
    logging_steps=1,
    save_strategy="no",
    bf16=False,
    seed=EXPERIMENT_SEED,
    data_seed=EXPERIMENT_SEED,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_policy_model,
    ref_model=dpo_reference_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)
dpo_trainer.train()

dpo_policy_model.save_pretrained(DPO_CHECKPOINT_DIR)
tokenizer.save_pretrained(DPO_CHECKPOINT_DIR)
write_artifact_manifest(
    stage="direct-preference-optimization-lora",
    output_dir=DPO_CHECKPOINT_DIR,
    training_files=ARIA_TRAIN_FILES,
    training_args=dpo_args,
    extra={
        "objective": "dpo",
        "rubric": "advance the scene over length-matched repetitive stalling",
        "training_pairs": len(preference_pairs),
        "sft_adapter": SFT_CHECKPOINT_DIR.relative_to(REPO_ROOT).as_posix(),
    },
)

instruct_lora_model = dpo_policy_model
del dpo_trainer, dpo_reference_model, dpo_reference_base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved DPO adapter and training-provenance manifest from the SFT checkpoint.")

### DPO Training Mechanism, Recapped

The complete training path now has four moving parts:

1. full-corpus contexts become fixed `prompt/chosen/rejected` triples;
2. chosen and rejected suffixes are length-matched so length is not an easy shortcut;
3. a frozen SFT reference anchors the starting behavior; and
4. the trainable policy receives the DPO update and is saved as a new adapter.

`beta` controls how strongly the pairwise update is regularized relative to the frozen reference. It is a training hyperparameter here; Part 3 explains how external evidence would guide its selection.

DPO still inherits the limits of its data: inconsistent labels, shallow style shortcuts, and missing preference categories remain training-data problems.

---

## Training Artifact Inventory

Part 1 writes three artifacts under `checkpoints/llm-finetuning/<profile>/`. Each artifact includes `experiment-manifest.json`, which records the model profile, pinned revision, seed, package versions, all 40 training chapter hashes, and training arguments.

| Artifact | Training signal | Mechanism demonstrated |
| --- | --- | --- |
| `non-instruction-full` | Raw Aria next-token prediction | Continued pretraining with all weights trainable |
| `instruction-lora` | One-sentence request/response demonstrations | Response-only SFT stored in LoRA adapters |
| `preference-dpo` | Length-matched advance-versus-stall comparisons | DPO against a frozen SFT reference |

An artifact proves that training state was saved. It does not prove quality, generalization, or release readiness. Those claims require the independent evaluation workflow in Part 3.

---

## Optional Reference: Resumable Full-Corpus Training Jobs

The practical objective lesson is complete above. The remaining cells package continued pretraining, SFT, and DPO as separate resumable jobs with explicit model handoffs and training-provenance manifests.

This is a training boundary, not a production release workflow. It deliberately does not calculate evaluation metrics, apply gates, rank candidates, or promote artifacts. Part 3 owns those responsibilities and requires an external benchmark that was not used to construct these training datasets.

The guarded runner remains disabled by default and rebuilds every objective from all 40 Aria chapters.

In [ ]:
from pathlib import Path
import gc

import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


def _resume_checkpoint(output_path):
    """Return the newest Trainer checkpoint in output_path, if one exists."""
    path = Path(output_path)
    return get_last_checkpoint(str(path)) if path.is_dir() else None


def _release_training_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_continued_pretraining(train_dataset, output_path, max_steps):
    """Run one resumable full-corpus CPT job from the selected plain base revision."""
    output_path = Path(output_path)
    model = AutoModelForCausalLM.from_pretrained(
        CPT_MODEL_NAME,
        revision=CPT_MODEL_REVISION,
    )
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=str(output_path),
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=5e-5,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            seed=EXPERIMENT_SEED,
            data_seed=EXPERIMENT_SEED,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        trainer.save_model(output_path)
        cpt_tokenizer.save_pretrained(output_path)
        write_artifact_manifest(
            "continued-pretraining-full-job",
            output_path,
            ARIA_TRAIN_FILES,
            args,
            model_name=CPT_MODEL_NAME,
            model_revision=CPT_MODEL_REVISION,
        )
        return {"artifact": str(output_path), "completed_steps": max_steps}
    finally:
        del trainer, model
        _release_training_memory()

### Resumable SFT Adapter Job

The SFT job writes a LoRA adapter and a training-provenance manifest while keeping the pinned base revision separate. Evaluation and release publication remain separate Part 3 workflows.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments


def run_lora_sft(train_dataset, output_path, max_steps):
    """Run one resumable full-corpus SFT job with a trainable LoRA adapter."""
    output_path = Path(output_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    model = get_peft_model(
        base_model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=LORA_TARGET_MODULES,
            bias="none",
        ),
    )
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=str(output_path),
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=2e-4,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            seed=EXPERIMENT_SEED,
            data_seed=EXPERIMENT_SEED,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        write_artifact_manifest(
            "supervised-fine-tuning-lora-job",
            output_path,
            ARIA_TRAIN_FILES,
            args,
            extra={"training_tasks": list(SFT_TRAIN_TASKS)},
        )
        return {"artifact": str(output_path), "completed_steps": max_steps}
    finally:
        del trainer, model, base_model
        _release_training_memory()

### Resumable DPO Adapter Job

DPO remains a separate auditable training job from the SFT adapter. The output manifest records the frozen SFT artifact, `beta`, complete training corpus, and chosen/rejected rubric. Part 3 owns any preference evaluation or promotion decision.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer


def run_dpo(train_dataset, sft_adapter_path, output_path, max_steps):
    """Run one resumable full-corpus DPO job against a frozen SFT reference."""
    sft_adapter_path = Path(sft_adapter_path)
    output_path = Path(output_path)
    policy_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    reference_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    policy = PeftModel.from_pretrained(
        policy_base,
        sft_adapter_path,
        is_trainable=True,
    )
    reference = PeftModel.from_pretrained(
        reference_base,
        sft_adapter_path,
        is_trainable=False,
    )
    reference.eval()
    for parameter in reference.parameters():
        parameter.requires_grad_(False)

    trainer = None
    try:
        args = DPOConfig(
            output_dir=str(output_path),
            per_device_train_batch_size=1,
            max_length=DPO_MAX_LENGTH,
            max_steps=max_steps,
            learning_rate=5e-5,
            beta=DPO_BETA,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            bf16=False,
            seed=EXPERIMENT_SEED,
            data_seed=EXPERIMENT_SEED,
            report_to="none",
        )
        trainer = DPOTrainer(
            model=policy,
            ref_model=reference,
            args=args,
            train_dataset=train_dataset,
            processing_class=tokenizer,
        )
        trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        policy.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        write_artifact_manifest(
            "direct-preference-optimization-lora-job",
            output_path,
            ARIA_TRAIN_FILES,
            args,
            extra={
                "sft_adapter": sft_adapter_path.relative_to(REPO_ROOT).as_posix(),
                "rubric": "advance the scene over length-matched repetitive stalling",
            },
        )
        return {"artifact": str(output_path), "completed_steps": max_steps}
    finally:
        del trainer, policy, reference, policy_base, reference_base
        _release_training_memory()

### Guarded Full-Corpus Job Runner

The disabled runner rebuilds each training dataset from all 40 Aria chapters and runs the three jobs in dependency order. It deliberately does not evaluate, gate, rank, or promote; those responsibilities belong to Part 3.

In [ ]:
import math

RUN_ARIA_JOB_SKELETON = False
ARIA_JOB_EPOCHS = 1

aria_job_training_results = {}
if RUN_ARIA_JOB_SKELETON:
    job_root = PROFILE_CHECKPOINT_DIR / "jobs" / "aria-objectives"
    continued_path = job_root / "continued-pretraining"
    sft_path = job_root / "sft-lora"
    dpo_path = job_root / "dpo"

    job_paragraphs = load_paragraphs(ARIA_TRAIN_FILES)
    job_causal = Dataset.from_dict({"text": job_paragraphs}).map(
        lambda examples: tokenize_causal(examples, cpt_tokenizer),
        batched=True,
        remove_columns=["text"],
    )

    job_instruction_pairs = build_instruction_pairs(
        ARIA_TRAIN_FILES,
        SFT_TRAIN_TASKS,
    )
    job_sft = Dataset.from_list(job_instruction_pairs).map(
        tokenize_instruction,
        remove_columns=["instruction", "response"],
    )

    job_preference_pairs = build_preference_pairs(
        ARIA_TRAIN_FILES,
        max_pairs=32,
    )
    job_dpo = Dataset.from_list(job_preference_pairs)

    print(
        f"Aria job input: {len(ARIA_TRAIN_FILES)} training chapters | "
        f"causal chunks={len(job_causal):,}, SFT pairs={len(job_sft):,}, "
        f"DPO pairs={len(job_dpo):,}"
    )

    continued_steps = ARIA_JOB_EPOCHS * math.ceil(len(job_causal))
    sft_steps = ARIA_JOB_EPOCHS * math.ceil(len(job_sft))
    dpo_steps = ARIA_JOB_EPOCHS * len(job_dpo)

    aria_job_training_results["continued_pretraining"] = run_continued_pretraining(
        train_dataset=job_causal,
        output_path=continued_path,
        max_steps=continued_steps,
    )
    aria_job_training_results["lora_sft"] = run_lora_sft(
        train_dataset=job_sft,
        output_path=sft_path,
        max_steps=sft_steps,
    )
    aria_job_training_results["dpo"] = run_dpo(
        train_dataset=job_dpo,
        sft_adapter_path=sft_path,
        output_path=dpo_path,
        max_steps=dpo_steps,
    )

aria_job_training_results

---

## End of Part 1: The Objective Axis

Each objective now has a complete-corpus, CPU-feasible teaching path:

| Objective | Example shape | Core intuition |
| --- | --- | --- |
| Continued pretraining | Raw paragraphs from all 40 chapters | Every real token practices the domain's next-token patterns |
| SFT | Request plus desired response | Prompt labels are masked; only the assistant suffix is supervised |
| DPO | Prompt plus chosen and rejected responses | A trainable policy learns from fixed comparisons against a frozen SFT reference |

The durable distinction is:

- raw text changes which sequences the model practices;
- demonstrations change which response contract it practices;
- preference pairs change which competing response it practices choosing.

Every checkpoint carries the pinned model revision, seed, package versions, complete training-corpus hashes, and training arguments. Continue to **[Part 2: Parameter-Based Techniques](02-llm-finetuning-parameter-techniques.ipynb)** for update-location intuition, then Part 3, where evaluation begins.